# 单图推理演示

中期验收第三项「能展示代码运行效果」的最小演示：加载最优权重，取一张 test 图，显示预测与置信度。

> 前提：先由本人本地执行 `python -m src.run --config configs/best.yaml` 生成 `outputs/best.pth`。
> 数据集仅用于教育研究，不用于临床；这里输出的只是模型概率，不构成任何诊断结论。


In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from src.utils import load_config
from src.model import build_model
from src.data import get_loaders

cfg = load_config('configs/best.yaml')
ckpt = 'outputs/best.pth'
assert os.path.isfile(ckpt), '请先运行 python -m src.run --config configs/best.yaml 生成权重'

state = torch.load(ckpt, map_location='cpu', weights_only=False)
model = build_model(state.get('config', cfg))
model.load_state_dict(state['model'])
model.eval()
print('loaded:', ckpt, '| best val_auc:', round(state.get('val_auc', 0), 4))


In [ ]:
_, _, test_loader = get_loaders(cfg)
x, y = next(iter(test_loader))
with torch.no_grad():
    prob = torch.softmax(model(x[:1]), dim=1)[0, 1].item()

img = x[0, 0].numpy() * 0.5 + 0.5  # 反归一化，仅用于显示
plt.figure(figsize=(3, 3))
plt.imshow(img, cmap='gray')
plt.title('true=%d  P(pneumonia)=%.3f  pred=%d' % (int(y[0]), prob, int(prob >= 0.5)))
plt.axis('off')
plt.show()
